# Ejercicio 5: Espacio Vectorial

## Objetivo de la práctica
- Implementar un Sistema de Recuperación de Información completo, desde la lectura del corpus hasta la recuperación de resultados.

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus
2. Realiza las etapas de preprocesamiento sobre el corpus


In [ ]:
import kagglehub
import pandas as pd

# Descargar la última versión del dataset
path = kagglehub.dataset_download("gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects")

# Buscar el archivo CSV
import os
csv_path = None
for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith(".csv"):
            csv_path = os.path.join(root, file)
            break
    if csv_path:
        break

# Cargar el DataFrame
if csv_path:
    df = pd.read_csv(csv_path)
    print(f"Corpus cargado: {len(df)} filas")
    print(f"Columnas: {df.columns.tolist()}")
    # Mostrar un ejemplo
    print("\nEjemplo de texto:")
    print(df['text'].iloc[0][:300])
else:
    print("No se encontró el archivo CSV")

In [ ]:
# Extraer la columna de textos como lista
corpus = df['text'].tolist()

# Opcional: eliminar la columna de índice si no la necesitas
df = df.drop(columns=['Unnamed: 0'])

In [ ]:
import pandas as pd
import zipfile
import os

# Descargar dataset
kaggle.api.dataset_download_files('gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects', path='./wikipedia_corpus', unzip=True)

# Cargar CSV
csv_file = './wikipedia_corpus/wikipedia_text_corpus.csv'
df = pd.read_csv(csv_file)
corpus = df['text'].tolist()

## Parte 1: Recuperación con TF-IDF

### Actividad:
3. Obtén la representación vectorial de los documentos utilizando el modelo TF-IDF
4. A partir de un conjunto de 10 queries, verifica la recuperación del sistema

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re
import warnings
warnings.filterwarnings('ignore')

In [ ]:
stop_words = set(stopwords.words('spanish'))

def preprocess_text(text):
    """
    Función de preprocesamiento para textos:
    - Convertir a minúsculas
    - Eliminar caracteres especiales y números
    - Eliminar stopwords
    - Tokenizar
    """
    if not isinstance(text, str):
        text = str(text)
    
    # Convertir a minúsculas
    text = text.lower()
    
    # Eliminar caracteres especiales y números (solo letras y espacios)
    text = re.sub(r'[^a-záéíóúüñ\s]', '', text)
    
    # Tokenizar
    tokens = word_tokenize(text, language='spanish')
    
    # Eliminar stopwords y palabras cortas (menos de 2 caracteres)
    tokens = [token for token in tokens if token not in stop_words and len(token) > 2]
    
    return ' '.join(tokens)

## Parte 2: Recuperación con BM25

### Actividad:
5. Implementa un sistema de recuperación usando el modelo BM25.
6. Para el mismo conjunto de 10 queries, verifica la recuperación del sistema

In [ ]:

print("\n" + "="*60)
print("PARTE 2: SISTEMA DE RECUPERACIÓN CON BM25")
print("="*60)

# Preparar corpus tokenizado para BM25
print("Preparando corpus tokenizado para BM25...")
tokenized_corpus = [doc.split() for doc in processed_corpus if doc.strip()]

# Filtrar documentos vacíos
valid_indices = [i for i, doc in enumerate(processed_corpus) if doc.strip()]
valid_corpus = [corpus[i] for i in valid_indices]
valid_tokenized = tokenized_corpus

print(f"Documentos válidos para BM25: {len(valid_tokenized)}")

# Inicializar BM25
bm25 = BM25Okapi(valid_tokenized)

# Función para recuperar documentos con BM25
def search_bm25(query, top_k=5):
    """Recupera los top_k documentos más relevantes para una query usando BM25"""
    query_tokens = query.split()
    scores = bm25.get_scores(query_tokens)
    top_indices = scores.argsort()[-top_k:][::-1]
    return [(valid_indices[idx], scores[idx]) for idx in top_indices if scores[idx] > 0]

# Evaluar todas las queries con BM25
print("Resultados de búsqueda BM25:")
print("-"*60)

for i, (query, proc_query) in enumerate(zip(queries, processed_queries)):
    print(f"Query {i+1}: {query}")
    print(f"   (Procesada: {proc_query})")
    
    results = search_bm25(proc_query, top_k=3)
    
    if results:
        print(f"Top {len(results)} resultados:")
        for rank, (doc_idx, score) in enumerate(results, 1):
            print(f" {rank}. Documento {doc_idx+1} (score: {score:.2f})")
            print(f" {corpus[doc_idx][:120]}...")
    else:
        print("No se encontraron resultados")

## Parte 3: Comparación de resultados

### Actividad:
7. Verifica cuáles documentos son recuperados (y en qué orden) por cada modelo de recuperación 

In [ ]:
print("\n" + "="*60)
print("PARTE 3: COMPARACIÓN DE MODELOS")
print("="*60)

# Crear tabla comparativa para cada query
comparison_results = []

for i, (query, proc_query) in enumerate(zip(queries, processed_queries)):
    # Obtener resultados de ambos modelos
    tfidf_results = search_tfidf(proc_query, top_k=5)
    bm25_results = search_bm25(proc_query, top_k=5)
    
    # Extraer solo los índices
    tfidf_indices = [idx for idx, _ in tfidf_results]
    bm25_indices = [idx for idx, _ in bm25_results]
    
    comparison_results.append({
        'query_id': i+1,
        'query': query,
        'tfidf_top5': tfidf_indices,
        'bm25_top5': bm25_indices,
        'intersection': set(tfidf_indices) & set(bm25_indices)
    })

# Mostrar comparación detallada
print("TABLA COMPARATIVA DE RESULTADOS:")
print("="*80)

for res in comparison_results:
    print(f"Query {res['query_id']}: {res['query'][:50]}...")
    print(f"TF-IDF top 5: {[i+1 for i in res['tfidf_top5']]}")
    print(f"BM25 top 5:    {[i+1 for i in res['bm25_top5']]}")
    print(f"Documentos comunes: {[i+1 for i in res['intersection']]}")
    print(f"Similitud: {len(res['intersection'])}/5 documentos coinciden")

# Análisis estadístico
print(" " + "="*60)
print("ANÁLISIS ESTADÍSTICO DE COMPARACIÓN:")
print("="*60)

avg_intersection = np.mean([len(res['intersection']) for res in comparison_results])
print(f"Promedio de documentos comunes por query: {avg_intersection:.1f}/5")

# Mostrar mejor y peor caso
best_case = max(comparison_results, key=lambda x: len(x['intersection']))
worst_case = min(comparison_results, key=lambda x: len(x['intersection']))

print(f"Mejor coincidencia - Query {best_case['query_id']}: {len(best_case['intersection'])}/5 documentos comunes")
print(f"Query: {best_case['query'][:60]}...")
print(f"Peor coincidencia - Query {worst_case['query_id']}: {len(worst_case['intersection'])}/5 documentos comunes")
print(f"Query: {worst_case['query'][:60]}...")

# Análisis de órdenes de ranking
print("ANÁLISIS DE RANKING:")
print("-"*40)

for res in comparison_results[:3]:  # Mostrar primeros 3 queries como ejemplo
    print(f"\nQuery {res['query_id']}:")
    print(f"  TF-IDF ranking: {[i+1 for i in res['tfidf_top5']]}")
    print(f"  BM25 ranking:   {[i+1 for i in res['bm25_top5']]}")
    
    # Verificar si el documento 1 (más relevante) coincide
    if res['tfidf_top5'] and res['bm25_top5']:
        if res['tfidf_top5'][0] == res['bm25_top5'][0]:
            print(f"Primer resultado coincide entre ambos modelos")
        else:
            print(f"Diferente primer resultado - TF-IDF: Doc{res['tfidf_top5'][0]+1}, BM25: Doc{res['bm25_top5'][0]+1}")